## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions


In [55]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROK_API_KEY")

groq_api_key

In [56]:
from langchain_groq import ChatGroq

model=ChatGroq(model="gemma2-9b-it",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001B4330DBCD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B4349DE8D0>, model_name='gemma2-9b-it', model_kwargs={})

In [57]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is yash I'm a generative AI learner")])

AIMessage(content="Hello Yash! It's great to meet you.\n\nIt's wonderful that you're learning about generative AI. It's a fascinating field with a lot of potential. \n\nWhat are you most interested in learning about generative AI? \n\nPerhaps I can help you explore some concepts, provide examples, or answer any questions you have.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 23, 'total_tokens': 97, 'completion_time': 0.134545455, 'prompt_time': 0.001323992, 'queue_time': 0.251655907, 'total_time': 0.135869447}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--e1b77d69-11d0-467d-965f-33c525752b4c-0', usage_metadata={'input_tokens': 23, 'output_tokens': 74, 'total_tokens': 97})

In [58]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Yash and i'm a generative AI engineer."),
        AIMessage(content="Hello Yash\n\nIt's great to meet you! I'm glad you're interested in generative AI. It's a fascinating field with a lot of potential. \n\nWhat specifically are you learning about generative AI? Are there any particular areas that interest you the most? \n\nI'm happy to answer any questions you have or discuss different aspects of generative AI. "),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="You said your name is Yash, and you're a generative AI engineer! 😊  \n\nIs there anything else you'd like to tell me about yourself or your work? I'm always eager to learn more.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 124, 'total_tokens': 172, 'completion_time': 0.087272727, 'prompt_time': 0.003302813, 'queue_time': 0.250980396, 'total_time': 0.09057554}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--da70be91-3c6a-48fe-8afa-5338c2a121a2-0', usage_metadata={'input_tokens': 124, 'output_tokens': 48, 'total_tokens': 172})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [59]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [60]:
config={"configurable":{"session_id":"chat1"}}

In [61]:
with_message_history.invoke(
    [HumanMessage(content="Hi, what is my name")],
    config=config
)

AIMessage(content="As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to know! 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 15, 'total_tokens': 56, 'completion_time': 0.074545455, 'prompt_time': 0.001267441, 'queue_time': 0.252127468, 'total_time': 0.075812896}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--a1fda1f2-8cb6-479d-8239-31753c83b7b2-0', usage_metadata={'input_tokens': 15, 'output_tokens': 41, 'total_tokens': 56})

In [62]:
# change the config -> session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi, what is my name")],
    config=config1
)
response.content

"As a large language model, I have no memory of past conversations and do not know your name. If you'd like to tell me, I'm happy to know! 😊\n"

In [65]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Yash")],
    config=config
)
response.content

"Hi Yash, it's nice to meet you! \n\nIs there anything I can help you with today? 😊  \n"

In [66]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, what is my name")],
    config=config
)
response.content

'Hi Yash,  \n\nYou told me your name was Yash!  Do you have a question I can help you with? 😊 \n'